# Five animals with a small CNN

## 1. Objective and concepts
Train a small convolutional neural network after the five-class Logistic Regression baseline. Convolutions learn local image patterns; pooling reduces spatial size. A final linear layer returns five logits. Cross-entropy combines these logits with the labels during training. Softmax is applied when we need estimated probabilities.

## 2. Libraries and data
PyTorch defines the network, computes gradients and trains on Apple Silicon MPS when available. NumPy loads the prepared arrays, pandas preserves image IDs, and Matplotlib plots the learning curves. We reuse exactly the five-class baseline test images. Pillow already prepared both input sizes in notebook 01.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'common.py').exists() else Path.cwd().parent
assert (ROOT / 'common.py').exists(), 'Open this notebook from the project or notebooks folder.'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from common import LABELS, SEED, pixels
np.random.seed(SEED)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})

import torch, random, copy
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score
from common import small_cnn, evaluate, show_predictions
random.seed(SEED); torch.manual_seed(SEED); torch.set_num_threads(4)
device=torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print('Training device:',device)
manifest=pd.read_csv(ROOT / 'data/processed/split_manifest.csv')
arrays=np.load(ROOT / 'data/processed/images.npz')
assert np.array_equal(manifest.image_id.to_numpy(),arrays['image_ids'])
X=torch.from_numpy(arrays['pixels64'].transpose(0,3,1,2).copy()).float()/255.0
y=torch.tensor(manifest.label_id.to_numpy(),dtype=torch.long)
train=manifest.split.eq('train').to_numpy(); validation=manifest.split.eq('validation').to_numpy(); test=manifest.split.eq('test').to_numpy()
softmax_test=pd.read_csv(ROOT / 'outputs/softmax_predictions.csv').image_id.to_numpy()
assert np.array_equal(softmax_test,manifest.loc[test,'image_id'].to_numpy())
generator=torch.Generator().manual_seed(SEED)
train_loader=DataLoader(TensorDataset(X[train],y[train]),batch_size=64,shuffle=True,generator=generator,num_workers=0)
validation_loader=DataLoader(TensorDataset(X[validation],y[validation]),batch_size=128,num_workers=0)
model=small_cnn().to(device)
print(model)
print('Trainable parameters:',sum(p.numel() for p in model.parameters()))

## 3. Train the classifier
We use Adam with learning rate 0.001, weight decay 0.0001, and batches of 64. The limit is 20 epochs. Early stopping saves the lowest validation loss and stops after four epochs without an improvement of at least 0.001. The test set does not affect stopping. Dropout and weight decay reduce model flexibility; they cannot guarantee generalization.

The fixed seed controls initialization and batch order. MPS kernels may still vary between runs, so reproducibility means preserving the exact splits, settings, versions and saved weights; bit-for-bit training equality across hardware is not promised.

In [ ]:
optimizer=torch.optim.Adam(model.parameters(),lr=0.001,weight_decay=0.0001)
criterion=nn.CrossEntropyLoss()
def run_epoch(loader,training):
    model.train(training); total_loss=0.0; total_correct=0; total=0
    with torch.set_grad_enabled(training):
        for images,targets in loader:
            images,targets=images.to(device),targets.to(device)
            if training: optimizer.zero_grad(set_to_none=True)
            scores=model(images); loss=criterion(scores,targets)
            if training: loss.backward(); optimizer.step()
            total_loss+=loss.item()*len(targets)
            total_correct+=(scores.argmax(1)==targets).sum().item(); total+=len(targets)
    return total_loss/total,total_correct/total
started=time.perf_counter(); best_loss=float('inf'); waiting=0; history=[]
for epoch in range(1,21):
    train_loss,train_accuracy=run_epoch(train_loader,True)
    val_loss,val_accuracy=run_epoch(validation_loader,False)
    history.append(dict(epoch=epoch,train_loss=train_loss,validation_loss=val_loss,
                        train_accuracy=train_accuracy,validation_accuracy=val_accuracy))
    print(f'Epoch {epoch:02d}: train loss {train_loss:.4f}, val loss {val_loss:.4f}, '
          f'train accuracy {train_accuracy:.3f}, val accuracy {val_accuracy:.3f}',flush=True)
    pd.DataFrame(history).to_csv(ROOT / 'outputs/cnn_history.csv',index=False)
    if val_loss<best_loss-0.001:
        best_loss=val_loss; best_epoch=epoch; waiting=0
        best_state={key:value.detach().cpu().clone() for key,value in model.state_dict().items()}
    else: waiting+=1
    if waiting>=4:
        print('Early stopping'); break
if device.type=='mps': torch.mps.synchronize()
elapsed=time.perf_counter()-started
model.load_state_dict(best_state)
model_path=ROOT / 'models/cnn.pt'
torch.save({'state_dict':best_state,'labels':LABELS,'input_size':64,'seed':SEED,'best_epoch':best_epoch},model_path)
history=pd.DataFrame(history); history.to_csv(ROOT / 'outputs/cnn_history.csv',index=False)
print('Best epoch:',best_epoch,'Training seconds:',round(elapsed,2))

## 4. Inspect learning behaviour
The training accuracy above is measured during optimization with dropout active. Validation uses evaluation mode. Rising training accuracy while validation loss rises supports an overfitting concern. Low scores on both sets can indicate underfitting or weak inputs. We also calculate the selected checkpoint's training and validation scores in evaluation mode.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(10,3.7))
for ax,metric in zip(axes,['loss','accuracy']):
    ax.plot(history.epoch,history[f'train_{metric}'],label='Training')
    ax.plot(history.epoch,history[f'validation_{metric}'],label='Validation')
    ax.axvline(best_epoch,color='gray',linestyle='--',label='Saved epoch')
    ax.set(xlabel='Epoch',ylabel=metric.capitalize(),title=f'CNN {metric}'); ax.legend()
fig.tight_layout(); fig.savefig(ROOT / 'outputs/cnn_learning_curves.png',dpi=180); plt.show()
def probabilities(mask,network=model,where=device):
    network.eval(); parts=[]
    with torch.no_grad():
        for (images,) in DataLoader(TensorDataset(X[mask]),batch_size=128):
            parts.append(torch.softmax(network(images.to(where)),dim=1).cpu().numpy())
    return np.concatenate(parts)
behaviour=[]
for split,mask in [('train',train),('validation',validation)]:
    predicted=probabilities(mask).argmax(1)
    behaviour.append(dict(split=split,accuracy=accuracy_score(y[mask],predicted),
                          macro_f1=f1_score(y[mask],predicted,average='macro')))
behaviour=pd.DataFrame(behaviour); behaviour.to_csv(ROOT / 'outputs/cnn_behaviour.csv',index=False); display(behaviour)

## 5. Evaluate the saved model
We reload the checkpoint on CPU and compare probabilities with the training device. The CNN and Softmax baseline must have identical test IDs and true labels.

In [ ]:
probability=probabilities(test)
checkpoint=torch.load(model_path,map_location='cpu',weights_only=True)
reloaded=small_cnn(); reloaded.load_state_dict(checkpoint['state_dict']); reloaded.eval()
small_mask=np.zeros(len(manifest),dtype=bool); small_mask[np.flatnonzero(test)[:16]]=True
np.testing.assert_allclose(probabilities(small_mask,reloaded,torch.device('cpu')),probability[:16],rtol=1e-3,atol=1e-4)
metrics,report=evaluate('cnn',y[test].numpy(),probability,manifest.loc[test,'image_id'].to_numpy(),
                        LABELS,model_path,elapsed,{'device':str(device),'best_epoch':best_epoch,
                        'parameters':sum(p.numel() for p in model.parameters())})
display(pd.Series(metrics)); display(report)
show_predictions('cnn',manifest)

## 6. Compare measured results
Accuracy counts all correct predictions. Macro precision, recall and F1 weight each class equally. The binary task is easier to define and uses fewer classes, so its score should not be ranked directly against the five-class tasks. Saved-file size is measured from disk; runtime also depends on dependencies and input preparation. Logistic training time includes the complete three-fold C search; CNN time includes training and epoch validation.

In [ ]:
comparison=pd.DataFrame([json.loads((ROOT / f'outputs/{name}_metrics.json').read_text())['summary']
                         for name in ['binary','softmax','cnn']])
comparison['file_MiB']=comparison.file_bytes/(1024**2)
comparison.to_csv(ROOT / 'outputs/model_comparison.csv',index=False)
display(comparison[['model','accuracy','precision_macro','recall_macro','f1_macro','test_images','file_MiB','training_seconds']])
a=pd.read_csv(ROOT / 'outputs/softmax_predictions.csv'); b=pd.read_csv(ROOT / 'outputs/cnn_predictions.csv')
assert a[['image_id','truth']].equals(b[['image_id','truth']])
fig,axes=plt.subplots(1,2,figsize=(9,3.7))
five=comparison[comparison.model.isin(['softmax','cnn'])]
axes[0].bar(five.model,five.accuracy,color=['#38678d','#bd6a3b']); axes[0].set(ylim=(0,1),ylabel='Test accuracy')
axes[1].bar(five.model,five.f1_macro,color=['#38678d','#bd6a3b']); axes[1].set(ylim=(0,1),ylabel='Test macro-F1')
fig.suptitle('Five-class comparison on identical held-out images'); fig.tight_layout()
fig.savefig(ROOT / 'outputs/model_comparison.png',dpi=180); plt.show()

## 7. Observations and limitations
The CNN completed 20 epochs and saved epoch 19, the lowest validation-loss checkpoint under the 0.001 improvement rule. Early stopping did not trigger. In evaluation mode, checkpoint training accuracy was 70.93% and validation accuracy was 62.97%, a 7.97 percentage-point gap. Test accuracy was 63.27% and macro-F1 was 0.586.

On identical test images, the CNN exceeded five-class Logistic Regression accuracy by 15.79 percentage points. The saved CNN is 361,413 bytes; the Softmax pipeline is 72,117 bytes. Both are small, but weight size does not include runtime dependencies. The CNN also receives 64 by 64 inputs while Logistic Regression receives 24 by 24, so this comparison does not isolate architecture alone.

There is no pretrained model or extensive search. Source identities are unavailable, near-duplicate detection is incomplete, and labels were sample-reviewed. Probabilities are uncalibrated. An unsupported animal is still forced into one of the five classes. No independent real-world collection was tested.

A review after evaluation found a Horse photograph with a Dog dataset label (ID `71c888b97f4ee9528f32`). The CNN predicts Horse, which counts as wrong against the saved label. We record this residual label noise without changing the completed test set or its reported scores.